# Your Voice Generator

Record 20 seconds of yourself talking. This turns your scripts into narration in your own voice — one voice, or two for a podcast.

---

## Before anything else — turn the GPU on

At the top of this page click **Runtime** → **Change runtime type** → pick **T4 GPU** → **Save**.

It's free. Nothing below works without it.

---

## Then just work down the page

Each grey box below is a **step**. Hover over it and a ▶ play button appears on the left. Click it, wait for the green tick, move to the next one.

You never have to type a command. Step 4 is the only place you type anything at all — your script.

## Step 1 — Check the GPU is on

Click ▶ below. You want to see a card name like `Tesla T4`.

If it says **no GPU**, go back and do the Runtime step above.

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU is ON:  {name}  ({gb:.0f} GB)')
    print('\nGood. Go to Step 2.')
else:
    print('no GPU.')
    print('\nFix it: Runtime menu > Change runtime type > T4 GPU > Save.')
    print('Then run this step again.')

## Step 2 — Install

Click ▶ and leave it alone for about **4 minutes**.

You'll see a lot of text scroll past. That's normal. Wait for `INSTALL FINISHED`.

*(If a button appears asking to restart the session, click it, then run this step once more.)*

In [ ]:
import os

# MIT-licensed community preservation of Microsoft's withdrawn VibeVoice repo.
if not os.path.isdir('/content/VibeVoice'):
    !git clone --depth 1 -q https://github.com/vibevoice-community/VibeVoice.git /content/VibeVoice
!pip install -q -e /content/VibeVoice 2>&1 | tail -2

os.makedirs('/content/voices', exist_ok=True)
print('\nINSTALL FINISHED — go to Step 3.')

## Step 3 — Give it your voice

You need a clip of yourself talking. A voice memo or a video off your phone is fine.

**What makes this work:**

- About **20 seconds** of you talking without stopping
- **Quiet room** — no music, no TV, nobody else talking
- Read it **the way you narrate your videos**. If you mumble the sample, every episode comes out mumbled.

Click ▶, then click **Choose Files** and pick your clip.

**Doing two voices?** Run this step again afterwards — change `NAME` to the second person and `GENDER` to `woman` first.

In [ ]:
NAME = 'Joe'   #@param {type:"string"}
GENDER = 'man' #@param ["man", "woman"]
#@markdown Skip the first few seconds (useful if the clip starts with silence):
START_AT_SECOND = 0 #@param {type:"number"}

import glob, os, subprocess
from google.colab import files
from IPython.display import Audio, display

print('Pick your audio or video clip...')
uploaded = files.upload()
source = '/content/' + list(uploaded.keys())[0]
out = f'/content/voices/en-{NAME}_{GENDER}.wav'

# 24 kHz mono is what the model expects; loudnorm evens out phone recordings.
subprocess.run([
    'ffmpeg', '-y', '-loglevel', 'error',
    '-ss', str(START_AT_SECOND), '-t', '20', '-i', source,
    '-vn', '-ar', '24000', '-ac', '1', '-c:a', 'pcm_s16le',
    '-af', 'highpass=f=70,loudnorm=I=-18:TP=-2:LRA=11', out,
], check=True)

print(f'\nSaved {NAME}. Listen back — this is what it will copy:')
display(Audio(out))
print('Voices ready:', [os.path.basename(v) for v in sorted(glob.glob('/content/voices/*.wav'))])

## Step 4 — Your script

Type or paste what you want said, between the `"""` marks below.

**One voice?** Just write normally, no labels needed.

**Two voices?** Start each line with `Speaker 1:` or `Speaker 2:` like the example.

Then click ▶. This step is instant — it only checks your script, it doesn't generate yet.

In [ ]:
script = """
Speaker 1: In 1961, a plane went down over Northern Rhodesia. On board was the Secretary-General of the United Nations.
Speaker 2: And the official verdict was pilot error.
Speaker 1: Pilot error. That was the finding. But three separate inquiries have reopened the case since.
"""

#@markdown Which voice reads which part. One name for a single narrator, two names for a dialogue:
VOICES = 'Joe Maya' #@param {type:"string"}

import glob, os, re

names = VOICES.split()
found = sorted(glob.glob('/content/voices/*.wav'))
paths = []
for n in names:
    hit = [f for f in found if f'-{n}_'.lower() in f.lower()]
    if not hit:
        raise SystemExit(
            f"No voice called '{n}'. You have: "
            f"{[os.path.basename(f) for f in found]}. Go back to Step 3."
        )
    paths.append(hit[0])

lines = []
for line in script.strip().splitlines():
    line = re.sub(r'[\[\(][^\]\)]{0,120}[\]\)]', ' ', line)   # drop [stage directions]
    line = re.sub(r'[*_`#]+', '', line).strip()               # drop markdown
    if not line:
        continue
    if not re.match(r'Speaker\s+\d+\s*:', line, re.I):
        line = f'Speaker 1: {line}'
    lines.append(line)

full_script = '\n'.join(lines).replace('\u2019', "'")
words = len(full_script.split())

used = {int(m) for m in re.findall(r'Speaker\s+(\d+)\s*:', full_script, re.I)}
if used and max(used) > len(names):
    raise SystemExit(
        f'Your script uses Speaker {max(used)} but you only listed '
        f'{len(names)} voice(s) in VOICES above.'
    )

print(f'Script looks fine: {words} words, roughly {words/150:.1f} minutes of audio.')
for i, (n, p) in enumerate(zip(names, paths), 1):
    print(f'  Speaker {i} = {n}')
print('\nThis is exactly what will be spoken:\n')
print(full_script)

## Step 5 — Make the audio

Click ▶ and wait.

**The first time** it downloads the model, so allow **5–10 minutes**. After that it's much quicker.

You'll get a player to listen to, and the file saves automatically to your computer.

In [ ]:
import time, torch
from google.colab import files
from IPython.display import Audio, display
from vibevoice.modular.modeling_vibevoice_inference import VibeVoiceForConditionalGenerationInference
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL = 'vibevoice/VibeVoice-1.5B'
#@markdown Higher = more expressive but less like your sample. Lower = steadier. 1.3 is a good start.
EXPRESSIVENESS = 1.3 #@param {type:"slider", min:1.0, max:2.0, step:0.1}

print('Loading the model (first run downloads it — be patient)...')
processor = VibeVoiceProcessor.from_pretrained(MODEL)
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map='cuda', attn_implementation='sdpa',
)
model.eval()
model.set_ddpm_inference_steps(num_steps=10)

print('Generating...')
started = time.time()
inputs = processor(
    text=[full_script], voice_samples=[paths],
    padding=True, return_tensors='pt', return_attention_mask=True,
)
inputs = {k: (v.to('cuda') if torch.is_tensor(v) else v) for k, v in inputs.items()}

outputs = model.generate(
    **inputs, max_new_tokens=None, cfg_scale=EXPRESSIVENESS,
    tokenizer=processor.tokenizer, generation_config={'do_sample': False},
    verbose=False, is_prefill=True,
)

processor.save_audio(outputs.speech_outputs[0], output_path='/content/episode.wav')
print(f'\nDONE in {(time.time()-started)/60:.1f} minutes.\n')
display(Audio('/content/episode.wav'))
files.download('/content/episode.wav')

---

## Doing another episode

Change the script in **Step 4**, run Step 4, then run Step 5. That's it — skip Steps 1–3, your voices are already loaded.

*(If you leave the tab for a while Colab disconnects and forgets everything. Then you start again from Step 1.)*

## If it doesn't sound right

| What's wrong | What to do |
|---|---|
| Doesn't sound like me | Record a better sample — quieter room, 20 seconds, your normal narration energy. Redo Step 3. |
| Flat and boring | Raise the **EXPRESSIVENESS** slider in Step 5 to 1.6–2.0 |
| Wobbly or slurred | Lower **EXPRESSIVENESS** to 1.1 |
| Talks too fast | Add full stops. Short sentences make it breathe. |
| Both voices sound the same | The two samples are too similar — use clearly different people |
| Crashes on a long script | Do it in halves and join them in your editor |

## A rule worth keeping

Only clone your own voice, or someone's who has actually said yes. If you're adding a second host, ask them first — Microsoft pulled this model's original release because people weren't doing that.